Group Pipeline — Diabetes and LifeStyle Dataset
IT2011 — Artificial Intelligence and Machine Learning — Progress Review I


Dataset: Diabetes and LifeStyle Dataset (97,297 rows x 31 columns)
Group task: integrate each member's individual preprocessing technique into a single,
logically-ordered pipeline, and produce the cleaned/engineered dataset used for modeling.

Each stage below re-runs that member's core logic against the previous stage's output, so the
full pipeline can be executed top-to-bottom in one run and produces the same
stage_.csv checkpoint files saved in results/outputs/.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE


In [2]:
RAW_PATH = 'data/raw/Diabetes_and_LifeStyle_Dataset_.csv'
df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)
df.head()

Raw shape: (97297, 31)


,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,52,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


Stage 1 — Missing Data (Rashmika)

In [3]:
print("Explicit NaNs:", df.isnull().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())

Explicit NaNs: 0
Exact duplicate rows: 0


In [4]:
clinical_cols = ['bmi', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
                  'triglycerides', 'glucose_fasting', 'glucose_postprandial', 'insulin_level',
                  'hba1c', 'systolic_bp', 'diastolic_bp', 'waist_to_hip_ratio']
implausible = sum((df[c] <= 0).sum() for c in clinical_cols)
print("Implausible (<=0) clinical readings:", implausible)

Implausible (<=0) clinical readings: 0


In [5]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after Stage 1:", df.shape)

Shape after Stage 1: (97297, 31)


Stage 2 — Encoding Categorical Variables (hiruni kawya)

In [6]:
education_order = {'No formal': 0, 'Highschool': 1, 'Graduate': 2, 'Postgraduate': 3}
income_order = {'Low': 0, 'Lower-Middle': 1, 'Middle': 2, 'Upper-Middle': 3, 'High': 4}
smoking_order = {'Never': 0, 'Former': 1, 'Current': 2}
diabetes_stage_order = {'No Diabetes': 0, 'Pre-Diabetes': 1, 'Gestational': 2, 'Type 2': 3, 'Type 1': 4}

df['education_level'] = df['education_level'].map(education_order)
df['income_level'] = df['income_level'].map(income_order)
df['smoking_status'] = df['smoking_status'].map(smoking_order)
df['diabetes_stage_encoded'] = df['diabetes_stage'].map(diabetes_stage_order)

nominal_cols = ['gender', 'ethnicity', 'employment_status']
df = pd.get_dummies(df, columns=nominal_cols, prefix=nominal_cols, drop_first=False)

print("Shape after Stage 2:", df.shape)
assert df.isnull().sum().sum() == 0, "Encoding introduced unexpected NaNs"
df.head(3)

Shape after Stage 2: (97297, 41)


,Age,education_level,income_level,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,family_history_diabetes,...,gender_Other,ethnicity_Asian,ethnicity_Black,ethnicity_Hispanic,ethnicity_Other,ethnicity_White,employment_status_Employed,employment_status_Retired,employment_status_Student,employment_status_Unemployed
0,58,1,1,0,0,215,5.7,7.9,7.9,0,...,False,True,False,False,False,False,True,False,False,False
1,52,1,2,1,1,143,6.7,6.5,8.7,0,...,False,False,False,False,False,True,True,False,False,False
2,60,1,2,0,1,57,6.4,10.0,8.1,1,...,False,False,False,True,False,False,False,False,False,True


Stage 3 — Outlier Treatment (laksith)

In [7]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

numeric_check_cols = ['bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate',
                       'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides',
                       'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c',
                       'alcohol_consumption_per_week', 'physical_activity_minutes_per_week']

for col in numeric_check_cols:
    low, high = iqr_bounds(df[col])
    df[col] = df[col].clip(lower=low, upper=high)

print("Shape after Stage 3 (capping, no rows dropped):", df.shape)

Shape after Stage 3 (capping, no rows dropped): (97297, 41)


Stage 4 — Normalization / Scaling (faij ahamed)

In [14]:
leakage = ['diagnosed_diabetes', 'diabetes_risk_score']
scale_exclude = leakage + ['diabetes_stage_encoded']
for_loop= df.select_dtypes(include=[np.number]).columns
numeric_cols = [c for c in for_loop if c not in scale_exclude]

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
print(f"Scaled {len(numeric_cols)} numeric feature columns with StandardScaler")
df[numeric_cols].describe().T[['mean', 'std']].head()

Scaled 19 numeric feature columns with StandardScaler


,mean,std
Age,-1.314508e-17,1.000005
physical_activity_minutes_per_week,9.347613e-18,1.000005
diet_score,-9.639726e-18,1.000005
family_history_diabetes,-9.931839e-18,1.000005
hypertension_history,-1.986368e-17,1.000005
